In [1]:
import requests
import re
import numpy as np
from datetime import datetime, timedelta
from sklearn.cluster import KMeans


NESHAN_API_KEY = "service.76b078ee5cb74c0fa91d531e91b38b52"              # کلید نشان خود را قرار دهید
H = {"Api-Key": NESHAN_API_KEY }  
START = "شیراز، خیابان خاکشناسی"

ADDRS = [
    # --- ۱۰ آدرس دقیق پهنه غرب و شمال‌غرب (خوشه A) ---
    "شیراز، بلوار شریعتی، میدان احسان",             # معالی‌آباد
    "شیراز، بلوار شهید رجایی، میدان معلم",          # فرهنگ‌شهر
    "شیراز، بلوار شهید چمران، تقاطع ابیوردی",       # چمران
    "شیراز، خیابان قصردشت، چهارراه زرگری",          # قصردشت
    "شیراز، بلوار ستارخان، تقاطع مصلی",            # ستارخان
    "شیراز، خیابان عفیف آباد، تقاطع قصردشت",       # عفیف‌آباد
    "شیراز، بلوار پاسداران، میدان سنگی",            # قدوسی / زرهی
    "شیراز، بلوار امیرکبیر، چهارراه سرباز",          # امیرکبیر
    "شیراز، خیابان ملاصدرا، چهارراه نمازی",         # ملاصدرا
    "شیراز، بلوار پاسداران، خیابان مبعث",          # زرهی

    # --- ۱۰ آدرس دقیق پهنه مرکز، شمال و شرق (خوشه B) ---
    "شیراز، بلوار نصر، میدان نصر",                 # نصر
    "شیراز، بلوار مدرس، میدان ولیعصر",              # مدرس
    "شیراز، بلوار گلستان، میدان کلبه سعدی",         # گلستان
    "شیراز، بلوار ارم، میدان ارم",                 # ارم
    "شیراز، بلوار جمهوری اسلامی، میدان دانشجو",      # جمهوری
    "شیراز، خیابان کریم‌خان زند، چهارراه زند",       # مرکز / زند
    "شیراز، خیابان لطف‌علی خان زند، چهارراه مشیر",  # مرکز / بازار
    "شیراز، خیابان شهید مطهری، چهارراه زرگری",       # مطهری
    "شیراز، بلوار رحمت، تقاطع بنی‌هاشمی",          # رحمت
    "شیراز، خیابان شهید دستغیب، تقاطع پارامونت"      # دستغیب / سینماسعدی
]


def clean_address(addr):
    q = addr
    q = re.sub(r"،?\s*پلاک\s*\S+", "", q)
    q = re.sub(r"(نرسیده به|بعد از|نبش|روبروی)\s*", "", q)
    return q.strip(" ،")


def geocode(addr):
    q_clean = clean_address(addr)
    search_text = f"شیراز، {q_clean}"
    try:
        r = requests.get(
            "https://api.neshan.org/v4/geocoding", 
            headers=H, 
            params={"address": search_text}, 
            timeout=10
        ).json()
        
        if r.get("status") == "OK" and "location" in r:
            return r["location"]["y"], r["location"]["x"]
            
        if isinstance(r.get("text"), list) and len(r["text"]) > 0:
            loc = r["text"][0]["location"]
            return loc["y"], loc["x"]
    except Exception:
        pass
    
    # فقط در صورت قطعی اینترنت یا خطای شبکه
    return 29.6100, 52.5300


def split_clusters_kmeans_balanced(pts):
    # نقاط مقصد (بدون مبدأ)
    dest_pts = np.array(pts[1:])  # شامل طول و عرض جغرافیایی (Lat, Lon)
    
    # ۱. اجرای K-Means واقعی دو بعدی روی تمام نقاط
    kmeans = KMeans(n_clusters=2, random_state=42, n_init=10).fit(dest_pts)
    
    # ۲. محاسبه فاصله هر نقطه تا مرکز خوشه اول (Cluster 0)
    centers = kmeans.cluster_centers_
    distances_to_c0 = np.linalg.norm(dest_pts - centers[0], axis=1)
    
    # ۳. انتخاب ۱۰ نقطه‌ای که به مرکز خوشه اول نزدیک‌ترند برای خوشه اول
    # و ۱۰ نقطه باقی‌مانده برای خوشه دوم
    sorted_indices = np.argsort(distances_to_c0)
    
    c1_indices = (sorted_indices[:10] + 1).tolist()
    c2_indices = (sorted_indices[10:] + 1).tolist()
    
    return c1_indices, c2_indices

def get_travel_time(p1, p2):
    try:
        o = f"{p1[0]},{p1[1]}"
        d = f"{p2[0]},{p2[1]}"
        r = requests.get("https://api.neshan.org/v1/distance-matrix", headers=H, params={"type": "car", "origins": o, "destinations": d}, timeout=5).json()
        if "rows" in r and len(r["rows"]) > 0:
            dur = r["rows"][0]["elements"][0]["duration"]["value"]
            return int(dur)
    except Exception:
        pass
    return 600

def optimize_cluster_neshan(start_pt, cluster_indices, pts):
    cluster_pts = [pts[i] for i in cluster_indices]
    all_points = [start_pt] + cluster_pts
    waypoints = "|".join(f"{lat},{lon}" for lat, lon in all_points)
    
    url = "https://api.neshan.org/v2/trip"
    params = {"points": waypoints, "source": "first", "destination": "any", "roundtrip": "false"}
    
    ordered_indices = []
    try:
        res = requests.get(url, headers=H, params=params, timeout=10).json()
        if "waypoint_order" in res:
            ordered_indices = [cluster_indices[i] for i in res["waypoint_order"]]
        elif "points" in res:
            raw = [p["index"] - 1 for p in res["points"] if p.get("index", 0) != 0]
            ordered_indices = [cluster_indices[i] for i in raw]
    except Exception:
        pass

    if not ordered_indices:
        ordered_indices = cluster_indices

    durations = []
    cur_pt = start_pt
    for idx in ordered_indices:
        next_pt = pts[idx]
        durations.append(get_travel_time(cur_pt, next_pt))
        cur_pt = next_pt

    return ordered_indices, durations

def main():
    print("۱. دریافت مختصات جغرافیایی نقاط...")
    all_addrs = [START] + ADDRS
    pts = [geocode(a) for a in all_addrs]
    
    print("\n================ 📍 لیست کامل نقاط و مختصات واقعی ================")
    for idx, (addr, coord) in enumerate(zip(all_addrs, pts)):
        tag = "مبدأ" if idx == 0 else f"مقصد {idx}"
        print(f"[{tag}] {addr}")
        print(f"       🌐 مختصات (Lat, Lon): {coord[0]:.4f}, {coord[1]:.4f}\n")
    print("==================================================================\n")

    print("۲. خوشه‌بندی متوازن (شرق/غرب)...")
    c1, c2 = split_clusters_kmeans_balanced(pts)

    now = datetime.now()
    clusters = [
        ("خوشه اول (گروه A - پهنه غرب)", c1),
        ("خوشه دوم (گروه B - پهنه شرق/مرکز)", c2)
    ]

    for c_name, c_idx in clusters:
        print(f"\n================ {c_name} ================")
        route, durations = optimize_cluster_neshan(pts[0], c_idx, pts)
        
        t = now
        total_sec = 0
        for step, node_idx in enumerate(route):
            addr_name = ADDRS[node_idx - 1]
            coord = pts[node_idx]
            travel_sec = durations[step]
            total_sec += travel_sec
            t += timedelta(seconds=travel_sec)
            
            print(f"  📍 مقصد {step + 1}: {addr_name}")
            print(f"     🌐 مختصات: ({coord[0]:.4f}, {coord[1]:.4f})")
            print(f"     ⏱️  زمان رسیدن: {t:%H:%M} (زمان رانندگی: {travel_sec // 60} دقیقه)\n")

        print(f"📌 مجموع زمان خالص رانندگی: {total_sec // 60} دقیقه")

if __name__ == "__main__":
    main()

۱. دریافت مختصات جغرافیایی نقاط...

================ 📍 لیست کامل نقاط و مختصات واقعی ================
[مبدأ] شیراز، خیابان خاکشناسی
       🌐 مختصات (Lat, Lon): 29.6325, 52.5349

[مقصد 1] شیراز، بلوار شریعتی، میدان احسان
       🌐 مختصات (Lat, Lon): 29.6790, 52.4563

[مقصد 2] شیراز، بلوار شهید رجایی، میدان معلم
       🌐 مختصات (Lat, Lon): 29.6763, 52.4541

[مقصد 3] شیراز، بلوار شهید چمران، تقاطع ابیوردی
       🌐 مختصات (Lat, Lon): 29.6365, 52.5088

[مقصد 4] شیراز، خیابان قصردشت، چهارراه زرگری
       🌐 مختصات (Lat, Lon): 29.6377, 52.5026

[مقصد 5] شیراز، بلوار ستارخان، تقاطع مصلی
       🌐 مختصات (Lat, Lon): 29.6252, 52.5001

[مقصد 6] شیراز، خیابان عفیف آباد، تقاطع قصردشت
       🌐 مختصات (Lat, Lon): 29.6277, 52.5071

[مقصد 7] شیراز، بلوار پاسداران، میدان سنگی
       🌐 مختصات (Lat, Lon): 29.7536, 52.3977

[مقصد 8] شیراز، بلوار امیرکبیر، چهارراه سرباز
       🌐 مختصات (Lat, Lon): 29.6071, 52.5053

[مقصد 9] شیراز، خیابان ملاصدرا، چهارراه نمازی
       🌐 مختصات (Lat, Lon): 29.6260, 52.4959

[مقص